<a href="https://colab.research.google.com/github/Sakinat-Folorunso/OOU_CSC309_Artificial_Intelligence/blob/main/notebooks/CSC309_Week07_NLP1_QA_Spam_CA2_Student_Centred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 309 – Artificial Intelligence
**Week 7 Practical (PH):** NLP I — text pipeline, TF-IDF retrieval QA, syntax → logic, spam detection (CA2)

**Instructor:** Dr Sakinat Folorunso · Associate Professor of AI Systems · Department of Computer Sciences, Olabisi Onabanjo University (OOU), Ago-Iwoye, Ogun State, Nigeria  
**Course:** CSC 309 (2 units; LH 15, PH 45) · **Mode:** student-centred, hands-on, Google Colab  
**CCMAS learning outcomes covered:** LO7

> 🎓 **Instructor note** – this notebook is the *STUDENT (TODO cells are stubs)* version.


## How to use this notebook
1. Fill in the **Group Log**, then do the **Do Now** with your partner.
2. Run the **Setup** cell once (*Runtime → Run all* is safe: nothing here crashes if a TODO is still empty).
3. Work top to bottom. Only edit cells marked **`# TODO(Student)`**.
4. Each task ends with an **auto-check** (✅/❌). Keep going until it is ✅.
5. Finish with the **Reflection**. Fast finishers: try the **Extensions**.

6. **Read the comments.** Every line of code has a plain-English comment above it. Read the comment first, predict what the line does, then run the cell and check your prediction.
7. Hand in the **Assignment** (marks and deadline are in the notebook) after the lab.

*Every random draw uses `SEED = 42`, so your numbers should match your classmates' numbers.*


## Lab tasks at a glance

| Part | Lab task(s) you must complete |
|---|---|
| Part A — Text pipeline and TF-IDF from scratch | Task A1 — Implement `Tfidf` |
| Part B — Retrieval question answering | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part C — Syntax and semantics: from an English sentence to predicate logic | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part D — Spam detection (CA2 core) — *a little machine learning* | Task D1 — Implement the metrics yourself |
| **Assignment** | see **📝 Assignment / deliverables** near the end |
| **Reflection** | short written reflection at the end (5–7 sentences) |
| *Extensions* | optional, for fast finishers |

In [ ]:
#@title 🧑🏽‍🤝‍🧑🏾 Group Log (fill before you start)
# Form field: type your group members' names here
group_members = "Type names here"   #@param {type:"string"}
# Form field: assign roles (driver, navigator, recorder, presenter)
roles_notes   = "Driver / Navigator / Recorder / Presenter"  #@param {type:"string"}
# Print them so your notebook records who did what
print("👥 Group:", group_members); print("📝 Roles:", roles_notes)


In [ ]:
#@title 🔧 Setup (run once) — seed, logging, helpers
# Standard-library modules used by every lab (os, re, math, random, logging, time ...)
import sys, os, re, math, random, logging, subprocess, warnings, time, itertools
# NumPy: fast arrays and random numbers
import numpy as np
# One seed for the whole course so every run gives the same numbers
SEED = 42                                    # one seed for the whole course → reproducible results
# Seed both Python's and NumPy's random generators
random.seed(SEED); np.random.seed(SEED)
# Configure logging so messages print as "LEVEL | message"
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
# A named logger for this course
log = logging.getLogger("csc309")
# Hide noisy library warnings so output stays readable
warnings.filterwarnings("ignore")

def ensure(pip_name, import_name=None):
    """Import a package; pip-install it first if it is missing (Colab-friendly)."""
    # Try to import the library
    try:
        # Return the imported module
        return __import__(import_name or pip_name)
    # If it is not installed ...
    except ImportError:
        # ... tell the student we are installing it
        log.info("installing %s ...", pip_name)
        # ... and install it with pip
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Return the imported module
        return __import__(import_name or pip_name)

# List that stores every check result
_CHECKS = []
def check(name, condition, hint=""):
    """Auto-check. Never raises, so 'Run all' always finishes. Pass a lambda to catch your own bugs."""
    # Try to import the library
    try:
        # Run the check: call it if it is a function, otherwise use the value
        ok = bool(condition() if callable(condition) else condition)
        # Default: no error message
        err = ""
    # If the student's code crashes, we report instead of stopping the notebook
    except Exception as e:                    # student code may crash; report instead of stopping
        # Mark as failed and keep the error message
        ok, err = False, f"{type(e).__name__}: {e}"
    # Store the result
    _CHECKS.append((name, ok))
    # Print a ✅ or ❌ line with a hint on failure
    print(("✅ PASS       " if ok else "❌ TRY AGAIN  ") + name + ("" if ok else f"   → {err or hint}"))
    # Return whether the check passed
    return ok

# Print how many checks passed
def checks_summary():
    # Count the passed checks
    passed = sum(ok for _, ok in _CHECKS)
    # Print the total
    print(f"\nChecks passed: {passed}/{len(_CHECKS)}")

def mean_ci(x, conf=0.95):
    """Mean, sample std and a two-sided t-based confidence interval."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert to a float array and count the observations
    x = np.asarray(x, dtype=float); n = len(x)
    # Mean and sample standard deviation (n-1 divisor)
    m = x.mean(); s = x.std(ddof=1) if n > 1 else 0.0
    # Half-width of the 95% confidence interval from the t-distribution
    h = stats.t.ppf((1 + conf) / 2, n - 1) * s / math.sqrt(n) if n > 1 else 0.0
    # Return mean, SD, lower and upper confidence limits
    return m, s, m - h, m + h

# Print a one-line statistical summary
def report(label, x):
    # Compute the statistics
    m, s, lo, hi = mean_ci(x)
    # Print mean, SD, CI and n
    print(f"{label:<28} mean={m:8.2f}  sd={s:7.2f}  95% CI=[{lo:.2f}, {hi:.2f}]  (n={len(x)})")

def paired_wilcoxon(a, b):
    """Two-sided Wilcoxon signed-rank test for paired samples (same seeds/instances)."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert both samples to arrays
    a, b = np.asarray(a, float), np.asarray(b, float)
    # If the two samples are identical the test is undefined
    if np.allclose(a, b):
        # Report p = 1 (no difference)
        return 1.0
    # Paired Wilcoxon signed-rank test p-value
    return float(stats.wilcoxon(a, b).pvalue)

# Setup finished
print("✅ Setup complete — seed =", SEED)


### Learning objectives (with Bloom level)
| # | By the end of this lab you can… | Bloom | CCMAS LO |
|---|---|---|---|
| 1 | build a text-processing pipeline (tokenise → normalise → remove stop-words → vectorise) | Apply | LO7 |
| 2 | implement **TF-IDF** and cosine similarity from scratch and verify against scikit-learn | Apply | LO7 |
| 3 | build a **retrieval question-answering** system and evaluate it (accuracy, MRR) | Apply / Evaluate | LO7 |
| 4 | parse a sentence (**syntax**) and compute its **logical meaning** (semantics) compositionally | Analyse / Create | LO7 |
| 5 | train and evaluate a **spam detector** with proper splits, precision/recall/F1 and CV | Evaluate | LO7 |

**Levels of language analysis:** morphology → **syntax** (structure: *S → NP VP*) → **semantics** (meaning: $\forall x(\text{Student}(x)\rightarrow\text{Passes}(x))$) → pragmatics (use in context).

**TF-IDF.** For term $t$, document $d$, corpus of $N$ documents with document frequency $\mathrm{df}(t)$ (scikit-learn's smoothed form):
$$\mathrm{tfidf}(t,d)=\mathrm{tf}(t,d)\cdot\Big(\ln\frac{1+N}{1+\mathrm{df}(t)}+1\Big),\qquad \hat{\mathbf v}_d=\frac{\mathbf v_d}{\lVert\mathbf v_d\rVert_2},\qquad \cos(q,d)=\hat{\mathbf v}_q\cdot\hat{\mathbf v}_d .$$

**Do Now (3 min).** *"Bank"* in "river bank" vs "bank loan": which level of analysis resolves it? Which WhatsApp autocorrect error have you seen this week?

## Part A — Text pipeline and TF-IDF from scratch

In [ ]:
#@title 🧹 Tokenise + stop-words (same tokeniser as scikit-learn so we can verify)
# Stop words: very common words that carry little meaning
STOP = {"a","an","the","is","are","of","to","and","in","on","for","what","which","does","do","how","why","where","with","that","by","as","it","its","their","from","can","be","or","not","than","so","when","about","using","only","if","at","each","these","this"}

# Split text into lower-case words and drop stop words
def tokenize(text, stop=STOP):
    # Regex finds words of two or more characters
    toks = re.findall(r"(?u)\b\w\w+\b", text.lower())               # words of ≥ 2 characters
    # Keep only tokens that are not stop words
    return [t for t in toks if t not in stop]

# A mini-corpus of course facts to search
CORPUS = [
 "Artificial intelligence studies intelligent agents that perceive their environment and take actions to achieve goals.",
 "A rational agent selects the action that maximises its expected performance measure given its percept history.",
 "PEAS stands for performance measure, environment, actuators and sensors.",
 "Breadth-first search expands the shallowest node first and uses a FIFO queue.",
 "Depth-first search expands the deepest node first and uses a LIFO stack, so it needs little memory.",
 "Uniform-cost search expands the node with the lowest path cost and is optimal for positive step costs.",
 "A* search evaluates nodes with f(n) = g(n) + h(n), where g is the path cost so far and h is the heuristic estimate.",
 "An admissible heuristic never overestimates the true cost of reaching the goal.",
 "Combinatorial explosion means the number of states grows exponentially with the depth of the search.",
 "Predicate logic uses quantifiers, predicates and variables to represent facts about objects.",
 "Resolution refutation proves a query by adding its negation to the clauses and deriving the empty clause.",
 "A semantic network represents knowledge as nodes for concepts and labelled links for relations such as is-a and part-of.",
 "A frame groups the slots of a concept and lets instances inherit default values from their parent frame.",
 "An expert system has a knowledge base, an inference engine and an explanation facility.",
 "Forward chaining starts from known facts and applies rules to derive new conclusions.",
 "Backward chaining starts from a goal and works back to find facts that support it.",
 "The Turing test asks whether a judge can tell a machine from a human using only text conversation.",
 "Searle's Chinese Room argues that manipulating symbols by rules is not the same as understanding.",
 "TF-IDF weights a word by how often it appears in a document and how rare it is across all documents.",
 "Naive Bayes classifiers assume that features are conditionally independent given the class.",
]
# Show the tokens of one sentence
print(tokenize(CORPUS[6]))

### Task A1 — Implement `Tfidf`
`fit(docs)` learns the vocabulary (alphabetical order, like scikit-learn) and the idf vector; `transform(docs)` returns an $n\times |V|$ NumPy array of **L2-normalised** TF-IDF rows (unknown words are ignored). Your numbers must match `TfidfVectorizer` exactly.

In [ ]:
# TODO(Student): implement the Tfidf class
# Our own TF-IDF vectoriser (like scikit-learn's)
class Tfidf:
    # TODO: learn the vocabulary and idf from documents
    def fit(self, docs):
        # Vocabulary: term to column number
        self.vocab = {}            # term → column index (alphabetical)
        # Inverse document frequencies
        self.idf = None            # np.array of idf values
        # Return the fitted object
        return self
    # TODO: turn documents into TF-IDF vectors
    def transform(self, docs):
        # Placeholder: a matrix of zeros of the right shape
        return np.zeros((len(docs), len(self.vocab)))

In [ ]:
#@title ✅ Auto-check for Task A1 (compare with scikit-learn)
# scikit-learn's version for comparison
from sklearn.feature_extraction.text import TfidfVectorizer
# Reference matrix from scikit-learn
sk = TfidfVectorizer(stop_words=sorted(STOP)); Xsk = sk.fit_transform(CORPUS).toarray()
# Our matrix
mine = Tfidf().fit(CORPUS); X = mine.transform(CORPUS)
check("same vocabulary as scikit-learn", lambda: sorted(mine.vocab) == sorted(sk.vocabulary_))
check("TF-IDF matrix equals scikit-learn's (atol 1e-9)", lambda: X.shape == Xsk.shape and np.allclose(X, Xsk, atol=1e-9))
# A query to test retrieval
q = ["What does A* use to evaluate nodes?"]
check("query vector equals scikit-learn's", lambda: np.allclose(mine.transform(q), sk.transform(q).toarray(), atol=1e-9))

## Part B — Retrieval question answering
Answer a question by returning the corpus sentence with the highest cosine similarity. Evaluate on 12 questions with a known gold sentence. Report **top-1 accuracy** (Wilson 95% CI) and **mean reciprocal rank** $\mathrm{MRR}=\frac1n\sum_i 1/\mathrm{rank}_i$.

In [ ]:
# TODO(Student): answer(question, k) → list of the k most similar corpus sentences (use cosine similarity)
# TODO: return the k best-matching sentences
def answer(question, k=1):
    # Placeholder: empty list
    return []

### 🔍 Code walkthrough — two numbers for a retriever
**Top-1 accuracy** asks "was the right sentence ranked first?". **MRR** (mean reciprocal rank) gives partial credit: rank 2 scores 1/2, rank 3 scores 1/3. Because the evaluation set is small we report the **Wilson interval** with the accuracy; with $n$ this small the interval is wide, and that honesty is part of the result.

In [ ]:
#@title 📏 Evaluate the QA system (accuracy with CI, MRR)
# Evaluation set: questions with the index of the correct answer
EVAL = [("Which queue does breadth-first search use?", 3), ("What is the formula that A* uses to evaluate nodes?", 6),
        ("What does PEAS stand for?", 2), ("Which search needs little memory because it uses a stack?", 4),
        ("What is an admissible heuristic?", 7), ("How does resolution prove a query?", 10),
        ("What are the parts of an expert system?", 13), ("Where does backward chaining start?", 15),
        ("What does the Chinese Room argue?", 17), ("Why does the number of states grow so quickly with depth?", 8),
        ("What assumption does Naive Bayes make about features?", 19), ("Which search is optimal for positive step costs?", 5)]
# Ranks of the correct answers
ranks = []
# For each question and gold answer
for qtext, gold in EVAL:
    # Rank all documents
    ranked = answer(qtext, k=len(CORPUS))
    # Rank of the gold answer (worse than last if missing)
    ranks.append(ranked.index(CORPUS[gold]) + 1 if ranked and CORPUS[gold] in ranked else len(CORPUS) + 1)
# Top-1 accuracy and mean reciprocal rank (MRR)
ranks = np.array(ranks); acc = float((ranks == 1).mean()); mrr = float((1 / ranks).mean())
# SciPy for the CI
from scipy import stats
# 95% Wilson CI for top-1 accuracy
ci = stats.binomtest(int((ranks == 1).sum()), len(ranks)).proportion_ci(0.95, method="wilson")
# Print the metrics
print(f"top-1 accuracy = {acc:.2f}  (95% Wilson CI [{ci.low:.2f}, {ci.high:.2f}], n={len(ranks)})   MRR = {mrr:.3f}")
# Show the failures
for (qtext, gold), r in zip(EVAL, ranks):
    # Print each failure
    if r > 1: print(f"  ✗ rank {r}: {qtext}")
check("top-1 accuracy ≥ 0.75", lambda: acc >= 0.75, "check normalisation and idf")
check("MRR ≥ 0.85",            lambda: mrr >= 0.85)

**Discuss.** n = 12 gives a *wide* confidence interval — what does that say about claims like "our QA is 90% accurate"? What happens to a *bag-of-words* retriever when the question uses a synonym ("lineup" instead of "queue") or a Yorùbá/Pidgin paraphrase?

## Part C — Syntax and semantics: from an English sentence to predicate logic
We parse with a tiny **context-free grammar** and compute the meaning **compositionally** (Frege's principle): the meaning of a phrase is a function of the meanings of its parts.

```
S  → NP VP          NP → Det N | PN          VP → V | V NP
```
Meanings: `every` = $\lambda N.\lambda P.\forall x\,(N(x)\rightarrow P(x))$, `some` = $\lambda N.\lambda P.\exists x\,(N(x)\wedge P(x))$, `no` = $\lambda N.\lambda P.\neg\exists x\,(N(x)\wedge P(x))$, a proper name `Ada` = $\lambda P.P(Ada)$.

In [ ]:
#@title 🌳 Mini parser + semantics (complete the lexicon: add 'no' and the verb 'teaches')
# Counter for fresh variable names
_v = itertools.count(1)
# Generate x1, x2, ...
fresh = lambda: f"x{next(_v)}"

# Determiners: "every" and "some" build quantified formulas
DET  = {"every": lambda N, P: ("forall", (v := fresh()), ("implies", N(v), P(v))),
        "some":  lambda N, P: ("exists", (v := fresh()), ("and", N(v), P(v)))}
# Nouns become predicates
NOUN = {"student": lambda x: ("Student", x), "lecturer": lambda x: ("Lecturer", x), "course": lambda x: ("Course", x)}
# Proper names
PN   = {"ada": "Ada", "bola": "Bola"}
# Intransitive verbs
VI   = {"passes": lambda x: ("Passes", x), "attends": lambda x: ("Attends", x)}
# Transitive verbs (TODO: add "teaches")
VT   = {"likes": lambda x, y: ("Likes", x, y)}              # TODO: add "teaches"

def parse_np(toks):
    """NP → Det N | PN  → returns (quantifier-function Q, rest)."""
    # If a noun phrase starts with a determiner
    if toks[0] in DET:
        # Pick the determiner and noun
        det, noun = DET[toks[0]], NOUN[toks[1]]
        # Return a function waiting for the verb phrase, plus the remaining tokens
        return (lambda P, det=det, noun=noun: det(noun, P)), toks[2:]
    # If it starts with a name
    if toks[0] in PN:
        # Look up the name
        name = PN[toks[0]]
        # Return a function that applies the verb phrase to the name
        return (lambda P, name=name: P(name)), toks[1:]
    # Otherwise we cannot parse
    raise ValueError(f"cannot parse NP at {toks}")

# Parse a whole sentence
def parse_s(sentence):
    # Lower-case and split into words
    toks = sentence.lower().rstrip(".").split()
    # Parse the subject noun phrase
    subj, rest = parse_np(toks)
    # The next word is the verb
    verb = rest[0]
    # Intransitive: apply the subject to the verb
    if verb in VI: return subj(VI[verb])
    # Transitive verb
    if verb in VT:
        # Parse the object noun phrase
        obj, rest2 = parse_np(rest[1:])
        # Combine subject, object and verb into a formula
        return subj(lambda x: obj(lambda y: VT[verb](x, y)))
    # Unknown verb
    raise ValueError(f"unknown verb {verb}")

In [ ]:
#@title ✅ Auto-check for Part C (variable names are normalised before comparing)
# Rename variables x1, x2 to x so formulas can be compared
def norm(s): return re.sub(r"x\d+", "x", s)
# Print a formula as readable logic
def show_f(t):
    # Atoms print as themselves
    if isinstance(t, str): return t
    # Operator name
    op = t[0]
    # Negation
    if op == "not": return "¬" + show_f(t[1])
    # Binary connectives
    if op in ("and", "or", "implies"): return "(" + show_f(t[1]) + {"and": " ∧ ", "or": " ∨ ", "implies": " → "}[op] + show_f(t[2]) + ")"
    # Quantifiers
    if op in ("forall", "exists"): return ("∀" if op == "forall" else "∃") + t[1] + " " + show_f(t[2])
    # Predicates
    return op + "(" + ", ".join(show_f(a) for a in t[1:]) + ")"

# Test on five sentences
for s in ["Every student passes", "Ada attends", "Some lecturer likes Bola", "Every lecturer teaches some course", "No student passes"]:
    # Print the formula
    try: print(f"{s:<38} ⇒ {show_f(parse_s(s))}")
    # Print an error without crashing
    except Exception as e: print(f"{s:<38} ⇒ ✗ {type(e).__name__}: {e}")
# Expected formulas for checking
want = {"Every student passes": "∀x (Student(x) → Passes(x))", "Ada attends": "Attends(Ada)",
        "Every lecturer teaches some course": "∀x (Lecturer(x) → ∃x (Course(x) ∧ Teaches(x, x)))"}
check("'Every student passes' ⇒ ∀x(Student→Passes)", lambda: norm(show_f(parse_s("Every student passes"))) == norm(want["Every student passes"]))
check("'Ada attends' ⇒ Attends(Ada)",                lambda: show_f(parse_s("Ada attends")) == "Attends(Ada)")
check("'No student passes' ⇒ ¬∃x(Student ∧ Passes)", lambda: norm(show_f(parse_s("No student passes"))) == "¬∃x (Student(x) ∧ Passes(x))")
check("nested quantifiers: 'Every lecturer teaches some course'", lambda: parse_s("Every lecturer teaches some course")[0] == "forall" and parse_s("Every lecturer teaches some course")[2][2][0] == "exists")

**Take it further.** The output uses the *same tuple format* as the Week 5 prover — paste a result there to prove things from English-derived KBs. Notice that our fragment gives only the **surface-scope** reading of *"Every lecturer teaches some course"* ($\forall\exists$); the reading $\exists\forall$ ("one course taught by all") is a **scope ambiguity** that real NLU systems must resolve.

## Part D — Spam detection (CA2 core) — *a little machine learning*
**ML essentials.** *Supervised learning* fits a function $f:\mathbf x\mapsto y$ from labelled examples. Always: (1) **split** data into train/test *before* looking at test labels; (2) fit the vectoriser **on training data only** (avoid leakage); (3) report **precision** $=\frac{TP}{TP+FP}$, **recall** $=\frac{TP}{TP+FN}$, $F_1=\frac{2PR}{P+R}$ for the spam class (accuracy alone misleads when spam is rare).

The real **UCI SMS Spam Collection** (5,574 messages) is downloaded in Colab. If the download is blocked the notebook falls back to a small **synthetic** Nigerian-style SMS set so the lab still runs (*numbers will then be much easier — do not report them as real results*).

In [ ]:
#@title 📥 Load the SMS data (real if reachable, synthetic otherwise)
# Load a real SMS spam data set if possible
def load_sms():
    # Download tools
    import io, zipfile, urllib.request
    # Try the real data
    for url in ["https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip",
                "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"]:
        # Attempt the download
        try:
            # Fetch the file with a 10 s timeout
            raw = urllib.request.urlopen(url, timeout=10).read()
            # Open the zip in memory
            with zipfile.ZipFile(io.BytesIO(raw)) as z:
                # Read the text lines
                lines = z.read("SMSSpamCollection").decode("utf-8", "ignore").splitlines()
            # Split label and message
            rows = [l.split("\t", 1) for l in lines if "\t" in l]
            # Return messages, labels (1 = spam) and a description
            return [r[1] for r in rows], np.array([int(r[0] == "spam") for r in rows]), "UCI SMS Spam Collection (real)"
        # Download failed
        except Exception as e:
            # Log it
            log.warning("could not download %s (%s)", url.split('/')[2], type(e).__name__)
    # Fall back to synthetic data
    return synthetic_sms()

# Synthetic SMS generator for offline use
def synthetic_sms(n=900, seed=SEED, spam_rate=0.2):
    # Seeded generator
    rng = np.random.default_rng(seed)
    # Helper to pick a random item
    pick = lambda xs: xs[rng.integers(0, len(xs))]
    # Spam templates
    spam_t = ["Congratulations! You have won N{a} in the {b} promo. Click {l} to claim now",
              "URGENT: your {k} account is suspended. Send your BVN to {n} to reactivate",
              "FREE {g}! Text WIN to {s} now. T&Cs apply", "Earn N{a} daily from home. No experience needed. Reply YES",
              "You are a lucky winner of {g}. Call {n} immediately to claim your prize", "Loan approved N{a}! No collateral. Click {l}",
              "Dear customer your {k} ATM card is blocked, confirm PIN at {l}", "Hot singles near you. Reply YES to chat free",
              "Hello dear, I have a business proposal for you. Reply for details", "Your parcel is held. Pay N{a} clearing fee today",
              "Dear {k} customer, update your details to avoid account closure", "Make money fast with crypto, join my group now"]
    # Ham templates
    ham_t = ["Hi {p}, are we still meeting at {t} for the {c} practical?", "Please bring your {i} to the lab on {d}",
             "Call me when you get home", "Mum says dinner is ready, come now", "{p}, the CA scores for {c} are on the notice board",
             "Is the {c} lecture free today? I might be late", "You won the debate yesterday, well done {p}!",
             "Don't forget to click save before you submit the {c} assignment", "I will send the money for transport tomorrow {p}",
             "Prayer meeting holds at {t} on {d}", "Can you win us a seat near the front, the {c} class is full", "Click the link to join the {c} class on Zoom at {t}",
             "Your {b} data bundle is almost finished, dial *131# to check balance", "{k} alert: your salary has been credited, thank you for banking with us",
             "Reply YES if you will attend the {c} tutorial on {d}", "Urgent: the {c} test has moved to {d}, please confirm to {p}"]
    # Fill-in values
    fill = dict(a=["50,000", "500,000", "5,000,000"], b=["MTN", "Glo", "Airtel", "Mega"], l=["bit.ly/x1", "http://win-now.ng", "tiny.cc/claim"],
                k=["GTB", "Access", "Zenith", "UBA"], n=["08031234567", "09076543210"], g=["airtime", "iPhone 15", "data bundle"], s=["30890", "44100"],
                p=["Ada", "Tayo", "Bola", "Ife", "Sam"], t=["4pm", "9am", "noon"], c=["CSC309", "CSC301", "DTS211"], i=["laptop", "calculator", "ID card"], d=["Monday", "Friday", "Tuesday"])
    # Messages and labels
    texts, y = [], []
    # Make n messages
    for _ in range(n):
        # Choose spam or ham
        spam = rng.random() < spam_rate
        # Pick a template
        t = pick(spam_t if spam else ham_t)
        # Fill the placeholders and store
        texts.append(re.sub(r"\{(\w)\}", lambda m: pick(fill[m.group(1)]), t))
        # Label with 2% noise
        y.append(int(spam) if rng.random() > 0.02 else 1 - int(spam))          # 2% label noise, like real annotation
    # Return data, labels and a warning that it is synthetic
    return texts, np.array(y), "SYNTHETIC toy SMS set (not real data)"

# Load the data
texts, y, SOURCE = load_sms()
# Show the data source and class balance
print(f"{SOURCE}: {len(texts)} messages, spam rate = {y.mean():.1%}"); print("example spam:", next(t for t, l in zip(texts, y) if l)); print("example ham :", next(t for t, l in zip(texts, y) if not l))

### Task D1 — Implement the metrics yourself
Write `prf(y_true, y_pred)` returning `(precision, recall, f1)` for the spam class (label 1) from the confusion counts TP/FP/FN. Return 0.0 when a denominator is 0.

In [ ]:
# TODO(Student): precision, recall, F1 for the positive class
# TODO: precision, recall, F1
def prf(y_true, y_pred):
    # Placeholder zeros
    return (0.0, 0.0, 0.0)

In [ ]:
#@title ✅ Check, then train and compare two classifiers (repeated stratified 5-fold CV, 3 repeats)
# scikit-learn metric tools
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
# Tiny test labels and predictions
yt = np.array([1,1,1,0,0,0,0,1]); yp = np.array([1,1,0,0,1,0,0,0])
# Reference values to compare
ref = precision_recall_fscore_support(yt, yp, average="binary", zero_division=0)[:3]
check("prf matches scikit-learn", lambda: np.allclose(prf(yt, yp), ref))

# Pipeline: vectoriser + classifier
from sklearn.pipeline import make_pipeline
# scikit-learn's version for comparison
from sklearn.feature_extraction.text import TfidfVectorizer
# Naive Bayes classifier
from sklearn.naive_bayes import MultinomialNB
# Logistic regression classifier
from sklearn.linear_model import LogisticRegression
# Cross-validation helpers
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split

# F1 over repeated cross-validation
def cv_f1(make_model, texts, y):
    # Store each fold's F1
    f1s = []
    # 5 folds, 3 repeats, seeded
    for tr, te in RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED).split(texts, y):
        # Fit a fresh model on the training fold
        m = make_model().fit([texts[i] for i in tr], y[tr])
        # Evaluate on the test fold
        f1s.append(prf(y[te], m.predict([texts[i] for i in te]))[2])               # vectoriser is fitted INSIDE each fold ⇒ no leakage
    # Return all fold scores
    return np.array(f1s)

# Pipeline factory: TF-IDF with bigrams, then Naive Bayes
mk_nb = lambda: make_pipeline(TfidfVectorizer(min_df=1, ngram_range=(1, 2), sublinear_tf=True), MultinomialNB(alpha=0.1))
# Pipeline factory: TF-IDF with bigrams, then logistic regression
mk_lr = lambda: make_pipeline(TfidfVectorizer(min_df=1, ngram_range=(1, 2), sublinear_tf=True), LogisticRegression(max_iter=1000, C=10, class_weight="balanced"))
# Score both models
f_nb, f_lr = cv_f1(mk_nb, texts, y), cv_f1(mk_lr, texts, y)
# Report mean, SD and CI
report("Naive Bayes  F1(spam)", f_nb); report("LogReg       F1(spam)", f_lr)
# Paired Wilcoxon over folds (indicative only)
print("paired Wilcoxon over the 15 folds: p =", f"{paired_wilcoxon(f_nb, f_lr):.3f}", "(folds overlap → treat p as indicative only)")

# Hold-out split 80/20, stratified
Xtr, Xte, ytr, yte = train_test_split(texts, y, test_size=0.2, stratify=y, random_state=SEED)
# Fit and predict
model = mk_lr().fit(Xtr, ytr); pred = model.predict(Xte)
# Confusion matrix
print("\nHold-out confusion matrix [rows=true ham/spam, cols=pred ham/spam]:\n", confusion_matrix(yte, pred))
# Precision, recall, F1 on the hold-out set
p, r, f = prf(yte, pred); print(f"precision={p:.3f}  recall={r:.3f}  F1={f:.3f}")
# Collect the misclassified messages
errs = [(t, l) for t, l, q in zip(Xte, yte, pred) if l != q]
# Print how many and the first five
print(f"\n{len(errs)} errors. First 5:"); [print(f"  true={'spam' if l else 'ham '}: {t[:90]}") for t, l in errs[:5]]
check("LogReg F1(spam) mean over CV ≥ 0.85", lambda: f_lr.mean() >= 0.85)

## 📝 CA2 deliverables (10 marks)
| Criterion | Marks |
|---|---|
| `Tfidf`, `answer`, `prf` correct (auto-checks ✅) | 3 |
| **QA demo**: three well-answered questions of *your own* + one failure case with an explanation (synonym? negation? low-resource language?) | 2 |
| **Spam**: confusion matrix + 3-sentence *error analysis* (which messages fool the model, and why) | 2 |
| **One documented improvement** (character n-grams, `class_weight`, a URL/number feature, threshold tuning) with before/after F1 (mean ± sd over CV) | 2 |
| Parser: add `no` and `teaches` and translate two new sentences | 1 |

## Reflection
- Why must the TF-IDF vectoriser be fitted on the training data only?
- Which is worse for a student inbox: a false positive or a false negative? How would you change the decision threshold?

```text
Write your reflection here…
```

## Extensions
- Build a **bigram** model and estimate sentence probability with add-one smoothing.
- Add a rule that distinguishes *"You won the debate"* (ham) from *"You have won N500,000"* (spam).

In [ ]:
checks_summary()  # how many auto-checks have you passed?